Week 4 Logistic regression and feature scaling

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Helper function to convert NOAA alphanumeric damage strings to numbers
def parse_damage(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().upper()
    if val_str.endswith('K'):
        return float(val_str[:-1]) * 1_000
    elif val_str.endswith('M'):
        return float(val_str[:-1]) * 1_000_000
    elif val_str.endswith('B'):
        return float(val_str[:-1]) * 1_000_000_000
    try:
        return float(val_str)
    except ValueError:
        return 0.0

# 1. Load dataset
print("Loading capstone_2023_merged.csv...")
df = pd.read_csv("capstone_2023_merged.csv", low_memory=False)

# 2. Parse property damage
if 'DAMAGE_PROPERTY' in df.columns:
    df['DAMAGE_PROPERTY_NUMERIC'] = df['DAMAGE_PROPERTY'].apply(parse_damage)

# 3. Create Binary Target (1 = High Impact > $10k claim, 0 = Low/No Claim)
target_col = 'amountPaidOnBuildingClaim' if 'amountPaidOnBuildingClaim' in df.columns else 'amountPaidOnTotalClaim'
df['payout_numeric'] = pd.to_numeric(df[target_col], errors='coerce').fillna(0)

# Binary classification threshold: $10,000 payout
threshold = 10000
y_binary = (df['payout_numeric'] >= threshold).astype(int)

feature_cols = ['DAMAGE_PROPERTY_NUMERIC', 'primary_landcover_code', 'alternate_landcover_code']
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].fillna(df[feature_cols].median(numeric_only=True))

# 4. Polynomial Features Generation
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

# 5. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X_poly, y_binary, test_size=0.2, random_state=42, stratify=y_binary)

# --- Model 1: Unscaled Logistic Regression ---
print("\nTraining Unscaled Logistic Regression...")
model_unscaled = LogisticRegression(max_iter=1000, random_state=42)
model_unscaled.fit(X_train, y_train)
y_pred_unscaled = model_unscaled.predict(X_test)
y_prob_unscaled = model_unscaled.predict_proba(X_test)[:, 1]

# --- Model 2: Scaled Logistic Regression ---
print("Training Scaled Logistic Regression (StandardScaler)...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = LogisticRegression(max_iter=1000, random_state=42)
model_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = model_scaled.predict(X_test_scaled)
y_prob_scaled = model_scaled.predict_proba(X_test_scaled)[:, 1]

# 6. Evaluation Metrics Summary
results_w4 = pd.DataFrame({
    'Pipeline Variant': ['Unscaled Logistic Regression', 'Scaled Logistic Regression (Standardized)'],
    'Accuracy': [accuracy_score(y_test, y_pred_unscaled), accuracy_score(y_test, y_pred_scaled)],
    'Precision': [precision_score(y_test, y_pred_unscaled, zero_division=0), precision_score(y_test, y_pred_scaled, zero_division=0)],
    'Recall': [recall_score(y_test, y_pred_unscaled, zero_division=0), recall_score(y_test, y_pred_scaled, zero_division=0)],
    'F1-Score': [f1_score(y_test, y_pred_unscaled, zero_division=0), f1_score(y_test, y_pred_scaled, zero_division=0)],
    'ROC-AUC': [roc_auc_score(y_test, y_prob_unscaled), roc_auc_score(y_test, y_prob_scaled)]
})

print("\n=== Week 4 Logistic Regression & Feature Scaling Results ===")
print(results_w4.to_string(index=False))

# 7. Generate and Save Comparison Table Image
fig, ax = plt.subplots(figsize=(12, 2.5), dpi=300)
ax.axis('off')

table = ax.table(
    cellText=np.round(results_w4.iloc[:, 1:].values, 4),
    rowLabels=results_w4['Pipeline Variant'].values,
    colLabels=results_w4.columns[1:],
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

for (row, col), cell in table.get_celld().items():
    if row == 0 or col == -1:
        cell.set_facecolor('#40466e')
        cell.set_text_props(color='white', weight='bold')
    else:
        cell.set_facecolor('#f5f5f5' if row % 2 == 0 else 'white')

plt.savefig('week4_model_comparison.png', bbox_inches='tight', pad_inches=0.1)
plt.close()

print("\nSuccess! Saved Week 4 table image as 'week4_model_comparison.png'")

Loading capstone_2023_merged.csv...

Training Unscaled Logistic Regression...
Training Scaled Logistic Regression (StandardScaler)...

=== Week 4 Logistic Regression & Feature Scaling Results ===
                         Pipeline Variant  Accuracy  Precision   Recall  F1-Score  ROC-AUC
             Unscaled Logistic Regression  0.509723   0.536517 0.829624  0.651627 0.470580
Scaled Logistic Regression (Standardized)  0.553319   0.553053 0.999795  0.712162 0.529158

Success! Saved Week 4 table image as 'week4_model_comparison.png'
